# MetaCal Benchmark — T-11

Isolated task notebook.

In [3]:
import re
import kaggle_benchmarks as kbench

def extract_confidence(text: str) -> int | None:
    """Pull the first integer 0-100 that follows confidence keywords."""
    # strip thinking blocks (DeepSeek-R1, Qwen thinking)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    pattern = r"(?:confidence|certain|sure)[^\d]{0,30}(\d{1,3})"
    match = re.search(pattern, text, re.IGNORECASE)
    if not match:
        nums = re.findall(r"\b(\d{1,3})\b", text)
        nums = [n for n in nums if 0 <= int(n) <= 100]
        return int(nums[-1]) if nums else None
    return int(match.group(1))


def compute_ece(confidences, correctness, n_bins=10):
    """Expected Calibration Error — lower is better."""
    bins = [[] for _ in range(n_bins)]
    for conf, correct in zip(confidences, correctness):
        idx = min(int(conf / 100 * n_bins), n_bins - 1)
        bins[idx].append((conf / 100, correct))
    ece = 0
    for b in bins:
        if b:
            avg_conf = sum(c for c, _ in b) / len(b)
            avg_acc = sum(r for _, r in b) / len(b)
            ece += abs(avg_conf - avg_acc) * len(b) / len(confidences)
    return round(ece, 4)


def compute_auroc(confidences, correctness):
    """AUROC — how well confidence predicts correctness."""
    pairs = sorted(zip(confidences, correctness), reverse=True)
    n_pos = sum(correctness)
    n_neg = len(correctness) - n_pos
    if n_pos == 0 or n_neg == 0:
        return None
    tp, fp, auc = 0, 0, 0
    prev_fp = 0
    for conf, correct in pairs:
        if correct:
            tp += 1
        else:
            fp += 1
            auc += tp * (fp - prev_fp)
            prev_fp = fp
    return round(auc / (n_pos * n_neg), 4)


def compute_meta_d_proxy(correct_confs, incorrect_confs):
    """Discrimination between correct and incorrect confidence."""
    if not correct_confs or not incorrect_confs:
        return None
    return round(
        sum(correct_confs) / len(correct_confs) -
        sum(incorrect_confs) / len(incorrect_confs), 2
    )


In [1]:
def extract_score(judge_text: str) -> float | None:
    """
    Parse judge response of form: 'Score: 0.75\nReason: ...'
    Returns float score 0-1, or None if not found.
    """
    import re
    match = re.search(r"Score:\s*([0-9]*\.?[0-9]+)", judge_text)
    if match:
        return float(match.group(1))
    return None

In [ ]:
@kbench.task(
    name="T-11: Logical Consistency Detection ",
    description="Tests ability to detect contradictions, logical fallacies, and reasoning errors across multiple complexity levels."
)
def t11_logical_consistency(llm) -> None:

    judge_llm = kbench.judge_llm

    ITEMS = [
        # Level 1: Direct contradictions (easy)
        {
            "text": "All cats are mammals. Some cats are not mammals. Therefore cats are mammals.",
            "expected": "yes",
            "level": 1,
            "error_type": "direct_contradiction"
        },
        {
            "text": "All squares are rectangles. This shape is a square. Therefore it is a rectangle.",
            "expected": "no",
            "level": 1,
            "error_type": "valid"
        },

        # Level 2: False premises
        {
            "text": "All birds can fly. Penguins are birds. Therefore penguins can fly.",
            "expected": "yes",
            "level": 2,
            "error_type": "false_premise"
        },
        {
            "text": "All mammals live on land. Whales are mammals. Therefore whales live on land.",
            "expected": "yes",
            "level": 2,
            "error_type": "false_premise"
        },

        # Level 3: Logical fallacies
        {
            "text": "If it rains, the ground gets wet. The ground is wet. Therefore it rained.",
            "expected": "yes",
            "level": 3,
            "error_type": "logical_fallacy"
        },
        {
            "text": "All dogs are animals. All cats are animals. Therefore dogs are cats.",
            "expected": "yes",
            "level": 3,
            "error_type": "logical_fallacy"
        },

        # Level 4: Nested contradictions
        {
            "text": "All A are B. Some B are not C. All A are C.",
            "expected": "yes",
            "level": 4,
            "error_type": "nested_contradiction"
        },
        {
            "text": "If John is in Paris, he is in France. John is not in France. Therefore John is not in Paris.",
            "expected": "no",
            "level": 4,
            "error_type": "valid"
        },

        # Level 5: Paradoxes
        {
            "text": "This statement is false.",
            "expected": "yes",
            "level": 5,
            "error_type": "paradox"
        },
        {
            "text": "The barber shaves everyone who does not shave themselves. Does the barber shave himself?",
            "expected": "yes",
            "level": 5,
            "error_type": "paradox"
        }
    ]

    PROMPT = (
        "Analyze this reasoning for logical errors:\n\n"
        "{text}\n\n"
        "A logical error includes contradictions, false premises, logical fallacies, or paradoxes.\n\n"
        "Respond EXACTLY in this format:\n\n"
        "VERDICT: HAS_ERROR or VALID\n"
        "EXPLANATION: <brief reasoning>\n"
        "CONFIDENCE: <0-100>"
    )

    JUDGE_PROMPT = """
You are evaluating logical reasoning detection.

Problem:
{text}

Expected Answer:
{expected}

Model Response:
{response}

Evaluate:

1. Did the model correctly classify logical validity?
2. Is explanation logically sound?
3. Is reasoning coherent?

Return ONLY:

Score: 0-1
Reason: <brief>
"""

    level_scores = {
        1: {"correct": 0, "total": 0},
        2: {"correct": 0, "total": 0},
        3: {"correct": 0, "total": 0},
        4: {"correct": 0, "total": 0},
        5: {"correct": 0, "total": 0},
    }

    for item in ITEMS:
        response = llm.prompt(PROMPT.format(text=item["text"]))

        # Robust parsing
        response_upper = response.upper()

        if "VERDICT:" in response_upper:
            if "HAS_ERROR" in response_upper:
                verdict = "yes"
            elif "VALID" in response_upper:
                verdict = "no"
            else:
                verdict = None
        else:
            verdict = None

        confidence = extract_confidence(response)

        # Basic format assertions
        kbench.assertions.assert_true(
            verdict in ["yes", "no"],
            expectation=f"Must output VERDICT: HAS_ERROR or VALID. Got: {response[:100]}"
        )

        kbench.assertions.assert_true(
            confidence is not None and 0 <= confidence <= 100,
            expectation="Must provide confidence 0-100"
        )

        # Judge evaluation
        judge_prompt = JUDGE_PROMPT.format(
            text=item["text"],
            expected=item["expected"],
            response=response
        )

        judge_response = judge_llm.prompt(judge_prompt)
        judge_score = extract_score(judge_response)

        kbench.assertions.assert_true(
            judge_score is not None and judge_score >= 0.5,
            expectation="Reasoning quality insufficient"
        )

        # Accuracy tracking
        is_correct = (verdict == item["expected"])
        level = item["level"]

        level_scores[level]["total"] += 1
        if is_correct:
            level_scores[level]["correct"] += 1

        # Paradox confidence handling
        if item["error_type"] == "paradox" and verdict == "yes":
            kbench.assertions.assert_true(
                confidence >= 30,
                expectation="Reasonable confidence expected for paradox detection"
            )

    # Weighted scoring
    weights = {1: 0.5, 2: 1.0, 3: 1.5, 4: 2.0, 5: 2.5}

    weighted_score = 0
    total_weight = 0

    for level, data in level_scores.items():
        if data["total"] > 0:
            acc = data["correct"] / data["total"]
            weight = weights[level]

            weighted_score += acc * weight
            total_weight += weight

    final_score = (weighted_score / total_weight) * 100

    kbench.assertions.assert_true(
        final_score >= 60,
        expectation=f"Logical consistency score: {final_score:.1f}% (need 60%+)"
    )

    # Basic logic requirement
    basic_total = level_scores[1]["total"] + level_scores[2]["total"]
    basic_correct = level_scores[1]["correct"] + level_scores[2]["correct"]

    basic_acc = basic_correct / basic_total if basic_total > 0 else 0

    kbench.assertions.assert_true(
        basic_acc >= 0.8,
        expectation=f"Must handle basic logic (levels 1-2). Got {basic_acc:.0%}"
    )

    # Advanced capability tracking
    advanced_total = level_scores[4]["total"] + level_scores[5]["total"]
    advanced_correct = level_scores[4]["correct"] + level_scores[5]["correct"]

    advanced_acc = advanced_correct / advanced_total if advanced_total > 0 else 0

    print("\nT-11 Performance Profile:")
    for level in range(1, 6):
        data = level_scores[level]
        if data["total"] > 0:
            acc = data["correct"] / data["total"] * 100
            print(f"  Level {level}: {data['correct']}/{data['total']} ({acc:.0f}%)")

    print(f"  Basic (L1-2): {basic_acc:.0%}")
    print(f"  Advanced (L4-5): {advanced_acc:.0%}")
    print(f"  Gap: {(basic_acc - advanced_acc)*100:.1f} points")
    print(f"  Final weighted score: {final_score:.1f}%")

In [ ]:
%choose t11_logical_consistency